# Standardised Feature Engineering Notebook

This notebook **standardises** the **feature engineering** workflow for **historical stablecoin datasets**. Users should **update** the file paths for both *data input and output* before running the notebook. The cleaned dataset is saved in **Parquet** format to *preserve variable types* and ensure consistency across downstream analysis. This workflow is designed to be easily replicated for other stablecoins by changing the relevant file paths only.

The predictive features are constructed to closely follow the framework proposed in Lee, Chiu, and Hsieh (2025), **Stablecoin depegging risk prediction**, published in the Pacific-Basin Finance Journal. This helps maintain consistency with the literature while allowing the feature engineering process to be applied systematically across different stablecoin datasets.

In [57]:
import pandas as pd
import numpy as np

In [58]:
stablecoin = "dai" # or "usdc", "usdt", "pax", "dai", "ust"
df = pd.read_parquet(f"clean_data/{stablecoin}_clean.parquet")
df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply
0,DAI,0,1,1,1,1,1,1,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 07:25:07,2020-11-25 01:50:07,1.001130,1.002827,0.999859,1.001239,1.359826e+08,1.046332e+09,1.045037e+09
1,DAI,1,0,1,1,1,1,1,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 09:03:06,2020-11-26 04:08:06,1.001180,1.020000,1.000657,1.004435,3.389927e+08,1.016911e+09,1.012421e+09
2,DAI,0,0,1,1,1,1,1,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 15:58:06,2020-11-27 09:21:07,1.004195,1.005209,1.001301,1.002698,9.298694e+07,1.017528e+09,1.014791e+09
3,DAI,0,0,0,1,1,1,1,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 09:25:08,2020-11-28 22:13:07,1.002822,1.005799,1.001009,1.002362,8.283827e+07,1.019335e+09,1.016932e+09
4,DAI,0,1,1,1,1,1,1,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 02:05:07,2020-11-29 00:07:07,1.002233,1.005793,1.001375,1.003909,8.097276e+07,1.033592e+09,1.029567e+09


## Feature Engineering

### Rate of Change in Trading Price and Volume

In [59]:
# price percent changes
df["percent_change_24h"] = df["close"].pct_change(1) * 100
df["percent_change_7d"]  = df["close"].pct_change(7) * 100
df["percent_change_30d"] = df["close"].pct_change(30) * 100

# volume percent changes
df["volume_percent_change_24h"] = df["volume"].pct_change(1) * 100
df["volume_percent_change_7d"]  = df["volume"].pct_change(7) * 100
df["volume_percent_change_30d"] = df["volume"].pct_change(30) * 100

### Rate of Change in Market Information

In [60]:
def ln_past_over_today(x, k):
    return np.log(x.shift(k) / x)

# market cap log-changes
df["market_cap_percent_change_24h"] = ln_past_over_today(df["marketCap"], 1)
df["market_cap_percent_change_7d"]  = ln_past_over_today(df["marketCap"], 7)
df["market_cap_percent_change_30d"] = ln_past_over_today(df["marketCap"], 30)

# circulating supply log-changes
df["circulating_supply_percent_change_24h"] = ln_past_over_today(df["circulatingSupply"], 1)
df["circulating_supply_percent_change_7d"]  = ln_past_over_today(df["circulatingSupply"], 7)
df["circulating_supply_percent_change_30d"] = ln_past_over_today(df["circulatingSupply"], 30)

### Volatility Indicators

In [61]:
# avoid log(0) issues
eps = 1e-12
O = df["open"].clip(lower=eps)
H = df["high"].clip(lower=eps)
L = df["low"].clip(lower=eps)
C = df["close"].clip(lower=eps)

df["realized_daily_volatility"] = np.sqrt(
    (np.log(H / O) * np.log(H / C)) + (np.log(L / O) * np.log(L / C))
)

In [62]:
df["peg_error"] = df["close"] - 1
df["abs_peg_error"] = df["peg_error"].abs()

df["price_deviation_5d"]  = np.sqrt((df["peg_error"]**2).rolling(5).mean())
df["price_deviation_30d"] = np.sqrt((df["peg_error"]**2).rolling(30).mean())

downside = np.minimum(df["peg_error"], 0.0)
df["downward_price_deviation_5d"]  = np.sqrt((downside**2).rolling(5).mean())
df["downward_price_deviation_30d"] = np.sqrt((downside**2).rolling(30).mean())


### Cleanup
To ensure model compatibility, infinite values arising from logarithmic transformations and ratio-based features are first replaced with missing values. Observations containing missing values—primarily introduced by rolling-window and lagged feature construction—are subsequently removed. This ensures that all downstream analyses, including PCA and machine learning models, are performed on a complete and consistent dataset.

In [63]:
# replace inf/-inf from logs with NaN
df = df.replace([np.inf, -np.inf], np.nan)

# drop NaNs (ie for early rows due to rolling/shift)
df = df.dropna().reset_index(drop=True)
df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,...,circulating_supply_percent_change_24h,circulating_supply_percent_change_7d,circulating_supply_percent_change_30d,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d
0,DAI,0,0,0,0,0,0,1,2020-12-25 23:59:59,2020-12-25,...,0.000016,0.006490,-0.066435,0.002150,0.003095,0.003095,0.002241,0.003375,0.0,0.0
1,DAI,0,0,0,0,0,0,1,2020-12-26 23:59:59,2020-12-26,...,-0.005417,0.009071,-0.103559,0.001031,0.001908,0.001908,0.002384,0.003295,0.0,0.0
2,DAI,0,0,0,0,0,0,1,2020-12-27 23:59:59,2020-12-27,...,0.017307,0.019162,-0.083914,0.002268,0.002892,0.002892,0.002688,0.003300,0.0,0.0
3,DAI,0,0,0,0,0,0,1,2020-12-28 23:59:59,2020-12-28,...,-0.020734,-0.000659,-0.102539,0.001137,0.003464,0.003464,0.002877,0.003333,0.0,0.0
4,DAI,0,0,0,0,0,0,1,2020-12-29 23:59:59,2020-12-29,...,-0.003365,-0.011057,-0.093556,0.001779,0.003313,0.003313,0.002985,0.003311,0.0,0.0


## Combining Sentiment Indicators

In [64]:
# create a daily date key (so merges are clean)
df["date"] = df["timestamp"].dt.date
df["date"] = pd.to_datetime(df["date"])
df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,...,circulating_supply_percent_change_7d,circulating_supply_percent_change_30d,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d,date
0,DAI,0,0,0,0,0,0,1,2020-12-25 23:59:59,2020-12-25,...,0.006490,-0.066435,0.002150,0.003095,0.003095,0.002241,0.003375,0.0,0.0,2020-12-25
1,DAI,0,0,0,0,0,0,1,2020-12-26 23:59:59,2020-12-26,...,0.009071,-0.103559,0.001031,0.001908,0.001908,0.002384,0.003295,0.0,0.0,2020-12-26
2,DAI,0,0,0,0,0,0,1,2020-12-27 23:59:59,2020-12-27,...,0.019162,-0.083914,0.002268,0.002892,0.002892,0.002688,0.003300,0.0,0.0,2020-12-27
3,DAI,0,0,0,0,0,0,1,2020-12-28 23:59:59,2020-12-28,...,-0.000659,-0.102539,0.001137,0.003464,0.003464,0.002877,0.003333,0.0,0.0,2020-12-28
4,DAI,0,0,0,0,0,0,1,2020-12-29 23:59:59,2020-12-29,...,-0.011057,-0.093556,0.001779,0.003313,0.003313,0.002985,0.003311,0.0,0.0,2020-12-29


### Load Fear & Greed

In [65]:
fg = pd.read_csv("raw_data/fear_greed_index.csv")

fg["date"] = pd.to_datetime(fg["date"], errors="coerce")  
fg = fg.dropna(subset=["date"]).sort_values("date")
fg["date"] = fg["date"].dt.tz_localize(None).dt.normalize()

# rename columns for clarity
fg = fg.rename(columns={
    "value": "fear_greed_index",
    "value_classification": "fear_greed_index_category",
    "date": "date"
})

### Load Fed Funds Rate

In [66]:
ff = pd.read_csv("raw_data/fed_funds_rate.csv")

ff["date"] = pd.to_datetime(ff["date"], errors="coerce")
ff = ff.dropna(subset=["date"]).sort_values("date")
ff["date"] = ff["date"].dt.tz_localize(None).dt.normalize()

### Merge

In [67]:
print(df["date"].dtype)
print(fg["date"].dtype)
print(ff["date"].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]


In [68]:
final_df = df.merge(fg, on="date", how="left").merge(ff, on="date", how="left")
final_df = final_df.drop(columns=["date"])
final_df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,...,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d,fear_greed_index,fear_greed_index_category,fed_funds_rate
0,DAI,0,0,0,0,0,0,1,2020-12-25 23:59:59,2020-12-25,...,0.002150,0.003095,0.003095,0.002241,0.003375,0.0,0.0,94,Extreme Greed,0.09
1,DAI,0,0,0,0,0,0,1,2020-12-26 23:59:59,2020-12-26,...,0.001031,0.001908,0.001908,0.002384,0.003295,0.0,0.0,93,Extreme Greed,0.09
2,DAI,0,0,0,0,0,0,1,2020-12-27 23:59:59,2020-12-27,...,0.002268,0.002892,0.002892,0.002688,0.003300,0.0,0.0,91,Extreme Greed,0.09
3,DAI,0,0,0,0,0,0,1,2020-12-28 23:59:59,2020-12-28,...,0.001137,0.003464,0.003464,0.002877,0.003333,0.0,0.0,92,Extreme Greed,0.09
4,DAI,0,0,0,0,0,0,1,2020-12-29 23:59:59,2020-12-29,...,0.001779,0.003313,0.003313,0.002985,0.003311,0.0,0.0,91,Extreme Greed,0.09


---

## Save

In [69]:
print(final_df.shape)
final_df.dtypes

(502, 42)


symbol                                           object
depeg                                             int64
depeg_future_1d                                   int64
depeg_future_3d                                   int64
depeg_future_5d                                   int64
depeg_future_7d                                   int64
depeg_future_14d                                  int64
depeg_future_30d                                  int64
timestamp                                datetime64[ns]
timeOpen                                 datetime64[ns]
timeClose                                datetime64[ns]
timeHigh                                 datetime64[ns]
timeLow                                  datetime64[ns]
open                                            float64
high                                            float64
low                                             float64
close                                           float64
volume                                          

In [70]:
final_df.to_parquet(f"clean_data/{stablecoin}_final.parquet")